In [4]:
from youtube_comment_downloader import YoutubeCommentDownloader
import pandas as pd

# Menginisiasi objek downloader
downloader = YoutubeCommentDownloader()
url = "https://youtu.be/Pbj0TslYZUw"

take_comments = downloader.get_comments_from_url(url)
all_comments = []

print("Mulai mengambil data...")

# Looping untuk mengambil komentar
for i, comment in enumerate(take_comments):
    # Batasi 100 komentar agar efisien
    if i >= 100:
        break
    
    text = comment['text'].strip()
    if text:
        all_comments.append(text)

# Memasukkan ke dalam tabel Pandas
df = pd.DataFrame({'text': all_comments})

# Menyimpan ke folder data/ (mundur 1 direktori dari prototype/)
file_path = '../data/dataset_youtube.csv'
df.to_csv(file_path, index=False)

print(f"Selesai! {len(all_comments)} komentar berhasil disimpan di {file_path}")

Mulai mengambil data...
Selesai! 100 komentar berhasil disimpan di ../data/dataset_youtube.csv


In [5]:
# Membaca kembali file CSV yang baru saja dibuat
df_cek = pd.read_csv('../data/dataset_youtube.csv')

# Menampilkan 5 baris pertama
df_cek.head()

,text
0,"Hallo warga sipil sekalian, setelah hamir sebu..."
1,JADI PUNDIT BOLA AJA BRO. GAK USAH BAHAS SEJAR...
2,INI NIH MANUSIA SOK. MANUSIA YANG BLOKIR IG SA...
3,ohh ini kang zen😅😂
4,KDMP emg sampah bangett pake dana desa loh


In [6]:
import re
from collections import Counter

# 1. Membaca dataset hasil scraping
df_youtube = pd.read_csv('../data/dataset_youtube.csv')

# 2. Menggabungkan semua komentar menjadi satu teks panjang
semua_komentar = " ".join(df_youtube['text'].astype(str))

# 3. Membersihkan teks (Case folding & hapus tanda baca)
teks_bersih = re.sub(r'[^a-z\s]', '', semua_komentar.lower())

# 4. Memecah teks menjadi token kata
kata_kata = teks_bersih.split()

# 5. Menghitung frekuensi kata menggunakan struktur data Counter (Hash Map)
frekuensi_kata = Counter(kata_kata)

# 6. Menampilkan 30 kata yang paling sering muncul
print("=== TOP 30 KATA TERBANYAK DI DATASET ===")
for kata, jumlah in frekuensi_kata.most_common(30):
    print(f"{kata}: {jumlah} kali")

=== TOP 30 KATA TERBANYAK DI DATASET ===
di: 42 kali
yg: 41 kali
bang: 29 kali
gw: 28 kali
dan: 23 kali
yang: 20 kali
ini: 17 kali
gak: 15 kali
itu: 15 kali
ada: 14 kali
sama: 13 kali
mau: 13 kali
bahas: 12 kali
kdmp: 12 kali
lu: 12 kali
sekolah: 12 kali
nya: 12 kali
jadi: 11 kali
apa: 11 kali
aja: 10 kali
presiden: 10 kali
juga: 10 kali
bisa: 10 kali
mereka: 9 kali
pemerintah: 9 kali
ferry: 9 kali
tidak: 9 kali
buat: 8 kali
saya: 8 kali
anda: 8 kali


In [7]:
data_slang = {
    'singkatan': [
        'yg', 'bgt', 'bgtt', 'bangett', 'gk', 'gak', 'ga', 
        'gajelas', 'gaje', 'udh', 'dgn', 'kreen', 'nyesel', 
        'tp', 'tonton', 'gw', 'lu', 'aja', 'kalo', 'klo', 
        'wkwk', 'sy', 'dpt', 'emg', 'trus', 'ma', 'tu', 'bg'
    ],
    'asli': [
        'yang', 'banget', 'banget', 'banget', 'tidak', 'tidak', 'tidak', 
        'tidak jelas', 'tidak jelas', 'sudah', 'dengan', 'keren', 'menyesal', 
        'tetapi', 'nonton', 'saya', 'kamu', 'saja', 'kalau', 'kalau', 
        'tertawa', 'saya', 'dapat', 'memang', 'terus', 'sama', 'itu', 'bang'
    ]
}

# Mengubah menjadi DataFrame
df_slang = pd.DataFrame(data_slang)

# Menyimpan ke folder data/ sebagai file final
file_path = '../data/kamus_slang.csv'
df_slang.to_csv(file_path, index=False)

print(f"Beres! Kamus slang final dengan {len(data_slang['singkatan'])} kata sudah tersimpan di {file_path}")

Beres! Kamus slang final dengan 28 kata sudah tersimpan di ../data/kamus_slang.csv


In [8]:
%pip install PySastrawi

     -------------------------------------- 211.2/211.2 kB 1.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.2.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# 1. Menyiapkan Kamus Normalisasi (Dictionary)
df_kamus = pd.read_csv('../data/kamus_slang.csv')
kamus_slang = dict(zip(df_kamus['singkatan'], df_kamus['asli']))

# 2. Menyiapkan Mesin Sastrawi (Stopword & Stemmer)
stopword_remover = StopWordRemoverFactory().create_stop_word_remover()
stemmer = StemmerFactory().create_stemmer()

# 3. Fungsi Utama Pipeline (Ini yang akan dijelaskan di presentasi)
def proses_teks_lengkap(teks):
    # A. Case Folding
    teks = teks.lower()
    
    # B. Cleansing (Hapus URL & Tanda Baca)
    teks = re.sub(r'http\S+|www\S+|https\S+', '', teks, flags=re.MULTILINE)
    teks = re.sub(r'[^a-z\s]', '', teks)
    
    # C. Normalisasi Kamus Slang
    words = teks.split()
    words_baku = [kamus_slang[kata] if kata in kamus_slang else kata for kata in words]
    teks_normal = " ".join(words_baku)
    
    # D. Stopword Removal (Menghapus kata 'di', 'dan', 'yg', dll)
    teks_stopword = stopword_remover.remove(teks_normal)
    
    # E. Stemming (Menghapus imbuhan menjadi kata dasar)
    teks_final = stemmer.stem(teks_stopword)
    
    return teks_final

# === MARI KITA UJI COBA ===
teks_kotor = "Wah presiden dan pemerintah itu gak dengerin rakyat ya!! Gw nyesel aja milih lu, sekolah aja mahal bgt."
teks_bersih = proses_teks_lengkap(teks_kotor)

print("Teks Asli   :", teks_kotor)
print("Teks Bersih :", teks_bersih)

Teks Asli   : Wah presiden dan pemerintah itu gak dengerin rakyat ya!! Gw nyesel aja milih lu, sekolah aja mahal bgt.
Teks Bersih : presiden perintah dengerin rakyat sesal milih sekolah mahal banget
